 <center>
    <h1>Adult Income Prediction - Logistic Regression Finetuning</h2>
 </center>

**Author:** Emily (most code taken from `baseline.ipynb`)

### Task:
The prediction task is to determine whether a person makes over $50K a year.
Given that the class imbalance is not severe and that both class labels are equally important, it is common to use classification accuracy or classification error to report model performance on this dataset.

### Parameters:
* age: the age of an individual.
* workclass: a general term to represent the employment status of an individual.
* fnlwgt: final weight. This is the number of people the census believes the entry represents.
* education: the highest level of education achieved by an individual.
* education­num: the highest level of education achieved in numerical form.
* marital­status: marital status of an individual.
* occupation: the general type of occupation of an individual.
* relationship: represents what this individual is relative to others.
* race: Descriptions of an individual’s race.
* sex: the sex of the individual.
* capital­gain: capital gains for an individual.
* capital­loss: capital loss for an individual.
* hours­per­week: the hours an individual has reported to work per week.
* native­country: country of origin for an individual.

### Purpose of this Notebook:
An attempt to finetune the baseline's Logistic Regression model, as well as improve data cleaning, feature engineering, and evaluation metrics.


### Changes We Made to the Baseline:
- added more evaluation metrics (precision, recall, F1 score, PR AUC)
- removed an unnecessary transformation (1/fnlwgt does nothing important)
- added a new feature 'is-married', which made a minor improvement
- fixed class imbalance (stratify=y in data splitting, class_weight='balanced' in model training)
- dropped 'fnlwgt' as it had no significance as a feature, it was intended to be used as a sample_weight
- put the preprocessing within in the pipeline so data isn't encoded twice
- made the optimisation step score by roc_auc score, and simplified the grid search pairs
- made the optimised model actually use the best model from the optimisation step (baseline made a mistake here)
- added some visualisation of model results (heatmaps)

### Additional Fixes:
- updated some deprecated functions for the visuals (sns.distplot, sns.theme, etc)
- fixed the correlation heatmap to only use numeric fields (possibly just a python version issue from the baseline)

### Results: (see the end)
- slight decrease in accuracy and precision
- significant improvement in recall and PR AUC

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, precision_score, roc_auc_score, f1_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GridSearchCV

RANDOM_STATE = 1

## 2. Data loading and exploration

In [ ]:
data = pd.read_csv("adult_income_dataset.csv")
data

In [ ]:
# Represents the number of rows and columns respectively
data.shape

In [ ]:
[(col, data[col].nunique(), data[col].dtype) for col in data.columns]

In [ ]:
# fnlwgt contains a high percentage of unique values, may be dropped later
data['fnlwgt'].nunique()/data.shape[0]*100

In [ ]:
# Confirm mild class imbalance (24.1% to 75.9%), should use class_weight='balanced' in model training
data['income'].value_counts()
data['income'].value_counts(normalize=True) * 100

In [ ]:
data.describe()

## 3. Numerical Variables EDA & Visualisation

#### Age

In [ ]:
sns.histplot(data=data, x='age', kde=True, color='darkblue')
plt.xlabel('Age');

#### FNLWGT (Final Weight)

In [ ]:
sns.histplot(data=data, x='fnlwgt', kde=True, color='darkblue')
plt.xlabel('Final Weight');

#### Education Num

In [ ]:
sns.histplot(data=data, x='education.num', kde=True, color='darkblue')
plt.xlabel('Education Number');

#### Capital Gain

In [ ]:
sns.histplot(data=data, x='capital.gain', kde=True, color='darkblue')
plt.xlabel('Capital Gain');

#### Capital Loss

In [ ]:
sns.histplot(data=data, x='capital.loss', kde=True, color='darkblue')
plt.xlabel('Capital Loss');

We see that final weight, capital gain, capital loss are right-skewed.

#### Hours Worked Per Week

In [ ]:
sns.histplot(data=data, x='hours.per.week', kde=True, color='darkblue')
plt.xlabel('Number of Hours Worked per Week');

## 4. Categorical Variables EDA & Visualisation

#### Income

In [ ]:
data['income'] = data['income'].apply(lambda x:x.replace("<=50K", "0"))
data['income'] = data['income'].apply(lambda x:x.replace(">50K", "1"))
data['income'] = data['income'].astype(int)

In [ ]:
sns.displot(x = data['income']);

#### Workclass

In [ ]:
sns.barplot(x = 'workclass', y = 'income', data = data)
plt.xticks(rotation = 90);

#### Education

In [ ]:
sns.barplot(x = data['education'], y = data['income'])
plt.xticks(rotation = 90);

In [ ]:
# Not a single person attending only preschool has an income >50k
len(data[(data['education'] == 'Preschool') & (data['income'] == 1)])

#### Marital Status

Here, Married-civ-spouse = when spouse is civilian.  
Married-af-spouse = when spouse is in Armed Forces.

In [ ]:
sns.barplot(x = 'marital.status', y = 'income', data = data)
plt.xticks(rotation = 90);

In [ ]:
plt.figure(figsize = (14,5))
sns.countplot(x = 'marital.status', data = data)
plt.xticks(rotation = 90);

#### Relationship

In [ ]:
sns.barplot(x = 'relationship', y = 'income', data = data)
plt.xticks(rotation = 90);

In [ ]:
plt.figure(figsize = (14,5))
sns.countplot(x = 'relationship', data = data)
plt.xticks(rotation = 90);

#### Occupation

In [ ]:
sns.barplot(x = 'occupation', y = 'income', data = data)
plt.xticks(rotation = 90);

In [ ]:
plt.figure(figsize = (14,5))
sns.countplot(x = 'occupation', data = data)
plt.xticks(rotation = 90);

#### Sex

In [ ]:
sns.set_theme(style = 'darkgrid', font_scale = 1)
sns.barplot(x = 'sex', y = 'income', data = data);

#### Sex by Race

In [ ]:
sns.set_theme(style = 'darkgrid', font_scale = 1.5)
sns.catplot(x = 'sex', y = 'income', data = data, kind = 'bar', col = 'race', height = 5, aspect = 1);

#### Race

In [ ]:
plt.figure(figsize = (14,5))
sns.countplot(x = 'race', data = data)
plt.xticks(rotation = 90);

#### Native Country

In [ ]:
sns.set_theme(style = 'darkgrid', font_scale = 1)
plt.figure(figsize = (14,5))
sns.barplot(x = 'native.country', y = 'income', data = data)
plt.xticks(rotation = 90)
plt.title('Country-wise Income');

In [ ]:
plt.figure(figsize = (14,5))
sns.countplot(x = 'native.country', data = data)
plt.xticks(rotation = 90);

## 5. Feature Engineering

In [ ]:
# Specify the numerical and categorical variables
numerical_var = [col for col in data.columns if data[col].dtype != object]
categorical_var = [col for col in data.columns if data[col].dtype == object]
print(numerical_var)
print(categorical_var)

In [ ]:
# We should make groups of age such that each group contains significant amount of information
print(len(data[(data['age'] > 16) & (data['age'] <= 25)]))
print(len(data[(data['age'] > 25) & (data['age'] <= 32)]))
print(len(data[(data['age'] > 32) & (data['age'] <= 40)]))
print(len(data[(data['age'] > 40) & (data['age'] <= 50)]))
print(len(data[(data['age'] > 50)]))

In [ ]:
# Age-binning to reduce number of columns in the DataFrame
data.loc[(data['age'] > 16) & (data['age'] <= 25), 'age'] = 1
data.loc[(data['age'] > 25) & (data['age'] <= 32), 'age'] = 2
data.loc[(data['age'] > 32) & (data['age'] <= 40), 'age'] = 3
data.loc[(data['age'] > 40) & (data['age'] <= 50), 'age'] = 4
data.loc[data['age'] > 50, 'age'] = 5

In [ ]:
# Forming groups in education
data['education'] = data['education'].apply(lambda x: 'School' if x == '11th' or x == '7th-8th' or x == '10th' 
                                              or x == '5th-6th' or x == '9th' or x == '12th' or x == '1st-4th' 
                                              or x == 'Preschool' else x)
data['education'] = data['education'].apply(lambda x: 'Associate' if x == 'Assoc-acdm' or x == 'Assoc-voc' else x)
education_map = {'School':1,
             'HS-grad':2,
             'Some-college':3,
             'Bachelors':4,
             'Prof-school':5,
             'Associate':6,
             'Masters':7,
             'Doctorate':8}
data['education'] = data['education'].map(education_map)

In [ ]:
# Creating a new feature
data['capital_diff'] = abs((data['capital.gain'] - data['capital.loss']))

In [ ]:
# Simplifying marital status
data['marital.status'] = data['marital.status'].apply(lambda x: 'Prev-Married' if (x == 'Widowed' or x == 'Divorced' or x == 'Separated') else x)
data['marital.status'] = data['marital.status'].apply(lambda x: 'Married' if (x == 'Married-civ-spouse' or x == 'Married-spouse-absent' or x == 'Married-AF-spouse') else x)

# Creating another new feature
data['is_married'] = (data['marital.status'] == 'Married').astype(int)

In [ ]:
# Simplifying workclass
data['workclass'] = data['workclass'].apply(lambda x: 'No income' if x == 'Never-worked' or x == 'Without-pay' else x)

In [ ]:
# Dividing race into ['White', 'Other]
data['race'] = data['race'].apply(lambda x: 'Other' if x != 'White' else x)

In [ ]:
# Percentage of people with a '?' value in every column.
for col in data.columns:
    print((col, len(data[data[col] == '?'])/len(data[col])*100))

In [ ]:
# Converting '?' values to NaN and imputing with mode to preserve data. Risk of inaccuracy, but preserves data.
data[data == '?'] = np.nan
col_with_symbol = ['workclass', 'occupation', 'native.country']
for col in col_with_symbol:
    data[col].fillna(data[col].mode()[0], inplace=True)

In [ ]:
# Dividing native country into ['US', 'Other']
data['native.country'] = data['native.country'].apply(lambda x: 'Other' if x != 'United-States' else x)

### Handling Skewness

In [ ]:
data['capital.gain'] = np.log1p(data['capital.gain'])
data['capital.loss'] = np.log1p(data['capital.loss'])

# removing this increased roc_auc by 0.1. Most likely due to the large number of zeros in capital_diff being considered a constant.
#data['capital_diff'] = np.log1p(data['capital_diff'])

In [ ]:
# Update category_var and numeric_var since new features have been added
category_var = [col for col in data.columns if data[col].dtypes == object]
numerical_var = [col for col in data.columns if data[col].dtypes != object]

### Checking for High Correlation

In [ ]:
plt.figure(figsize = (10, 5))
# this threw an error in the original baseline because of the non-numeric columns, could be due to python version
# sns.heatmap(data.corr(), annot = True)

# Fix:
sns.heatmap(data[numerical_var].corr(), annot=True);

**Important:** 'education' and 'education.num' are highly correlated with each other. Therefore we should drop one of them in order to significance and uniqueness of the variable. We will see which one to drop using multi-collinearity.

### Multi-Collinearity

Multicollinearity is present if the feature's Variance Inflation Factor (VIF) score is > 10.

In [ ]:
def calc_vif(X):
    vif = pd.DataFrame()
    vif["variables"] = X.columns
    vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    return(vif)

In [ ]:
dt = data[numerical_var]
calc_vif(dt)

In [ ]:
# Deleting 'education.num' as it has a VIF of 29.16 compared to 'education's VIF score of 11.98 above
data = data.drop('education.num', axis = 1)
# Update numeric_var again
numerical_var = [col for col in data.columns if data[col].dtypes != object]

In [ ]:
# Confirm improvement in VIF scores after removing 'education.num'
dt = data[numerical_var]
calc_vif(dt)

# Now all VIF scores are below 10, so multi-collinearity has been resolved

## 6. Splitting the Data (80/20)

In [ ]:
# Separate labels from features
y = data['income']
X = data.drop(['income', 'fnlwgt'], axis = 1)
# Final weight is meant to be used as a sample weight, never as a feature, so we drop it

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=RANDOM_STATE, test_size=0.2, stratify=y) 

## 7. Building & Evaluating the Pipeline

In [ ]:
# Update category_var and numeric_var since features have been dropped
categorical_var = [col for col in X.columns if X[col].dtypes == object]
numerical_var = [col for col in X.columns if X[col].dtypes != object]
print(f'Numeric features: {numerical_var}')
print(f'Categorical features: {categorical_var}')

In [ ]:
# Shared preprocessor for both pipelines
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical_var),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_var)
])

# Base pipeline
base_pipe = Pipeline([
    ('pre', preprocessor),
    ('clf', LogisticRegression(class_weight='balanced', random_state=RANDOM_STATE, max_iter=200))
])

base_pipe.fit(X_train, y_train)

In [ ]:
pred_base = base_pipe.predict(X_test)
pred_base_proba = base_pipe.predict_proba(X_test)[:, 1]

In [ ]:
# Base Logistic Regression Model
acc = accuracy_score(y_test, pred_base)
prec = precision_score(y_test, pred_base)
rec = recall_score(y_test, pred_base)
f1 = f1_score(y_test, pred_base)
roc_auc = roc_auc_score(y_test, pred_base_proba)

print('Accuracy Score: {0:0.3f}'. format(acc))
print('Precision Score: {0:0.3f}'. format(prec))
print('Recall Score: {0:0.3f}'. format(rec))
print('F1 Score: {0:0.3f}'. format(f1))
print('ROC AUC Score: {0:0.3f}'. format(roc_auc))

In [ ]:
# Base Model Confusion Matrix
cm = confusion_matrix(y_test, pred_base)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, linewidths=1, linecolor='black', 
            xticklabels=['<=50K', '>50K'], yticklabels=['<=50K', '>50K'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix (Unoptimised LogReg)')
plt.show()

tn, fp, fn, tp = cm.ravel()
total = cm.sum()
print(f'True positives  (correctly predicted >50K): {tp} ({tp/total:.2%})')
print(f'True negatives  (correctly predicted <=50K): {tn} ({tn/total:.2%})')
print(f'False positives (predicted >50K, actually <=50K): {fp} ({fp/total:.2%})')
print(f'False negatives (predicted <=50K, actually >50K): {fn} ({fn/total:.2%})')

## 8. Optimization

In [ ]:
# New pipeline to be optimised, reusing original preprocessor
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])

# Create param grid. Split by penalty to avoid incompatible pairings, reduced options to decrease runtime.
param_grid = [
    # L2 (fast, strong baseline)
    {
        'classifier__penalty': ['l2'],
        'classifier__C': [0.01, 0.1, 1, 10],
        'classifier__solver': ['lbfgs', 'saga'],
        'classifier__class_weight': ['balanced'],
        'classifier__max_iter': [200]
    },

    # L1 (feature selection)
    {
        'classifier__penalty': ['l1'],
        'classifier__C': [0.01, 0.1, 1, 10],
        'classifier__solver': ['saga'],
        'classifier__class_weight': ['balanced'],
        'classifier__max_iter': [200]
    },

    # ElasticNet (controlled exploration)
    {
        'classifier__penalty': ['elasticnet'],
        'classifier__C': [0.01, 0.1, 1],
        'classifier__l1_ratio': [0.3, 0.7],
        'classifier__solver': ['saga'],
        'classifier__class_weight': ['balanced'],
        'classifier__max_iter': [200]
    }
]

# Create grid search object
clf = GridSearchCV(pipe, param_grid = param_grid, cv = 5, scoring='roc_auc', verbose=True, n_jobs=-1)

# Fit on data
best_clf = clf.fit(X_train, y_train)

In [ ]:
best_clf.best_params_

## 9. Evaluating Optimised Model

In [ ]:
# Optimal model results
best_model = best_clf.best_estimator_
pred_opt = best_model.predict(X_test)
pred_opt_proba = best_model.predict_proba(X_test)[:, 1]

opt_acc = accuracy_score(y_test, pred_opt)
opt_prec = precision_score(y_test, pred_opt)
opt_rec = recall_score(y_test, pred_opt)
opt_f1 = f1_score(y_test, pred_opt)
opt_roc_auc = roc_auc_score(y_test, pred_opt_proba)

print('Accuracy Score: {0:0.3f}'. format(opt_acc))
print('Precision Score: {0:0.3f}'. format(opt_prec))
print('Recall Score: {0:0.3f}'. format(opt_rec))
print('F1 Score: {0:0.3f}'. format(opt_f1))
print('ROC AUC Score: {0:0.3f}'.format(opt_roc_auc))

In [ ]:
# Confusion Matrix after optimisation
cm = confusion_matrix(y_test, pred_opt)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, linewidths=1, linecolor='black',
            xticklabels=['<=50K', '>50K'], yticklabels=['<=50K', '>50K'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix (Unoptimised LogReg)')
plt.show()

tn, fp, fn, tp = cm.ravel()
total = cm.sum()
print(f'True positives  (correctly predicted >50K): {tp} ({tp/total:.2%})')
print(f'True negatives  (correctly predicted <=50K): {tn} ({tn/total:.2%})')
print(f'False positives (predicted >50K, actually <=50K): {fp} ({fp/total:.2%})')
print(f'False negatives (predicted <=50K, actually >50K): {fn} ({fn/total:.2%})')

## 10. Comparison of Results

In [ ]:
results = pd.DataFrame({
    'Model'    : ['Baseline (original)', 'Baseline (improved)', 'Optimised'],
    'Accuracy' : [0.862, acc, opt_acc],
    'Precision': [0.769, prec, opt_prec],
    'Recall'   : [0.612, rec, opt_rec],
    'F1'       : [0.682, f1, opt_f1],
    'ROC AUC'  : [0.777, roc_auc, opt_roc_auc],
}).round(3)
print(results)